# Control 2: the same correlation analysis on random-grouping models

Reviewer E7Cp, follow-up comment (Nov 19, 2026):

> "Do the same correlation analysis on the random grouping models, using the
> gate of the group that prefers myeloid cells. If the random groups also give
> similar r values and similar top signatures, then the result in Table 9 is
> not related to the CGP knowledge itself. If the correlations are clearly
> lower, this would be good evidence that the CGP expert really carries the
> biology."

We take all six random-grouping models (3 random assignments x seeds 42/43;
same data, same training protocol, modules assigned to groups uniformly at
random). For each model we pick the gate that prefers myeloid cells (largest
myeloid-minus-lymphoid mean weight), then repeat the Table 9 analysis:
correlation of that gate's per-cell weight with every module's activity,
plus the partial correlation controlling for the myeloid label.

The real model is included for direct comparison (its myeloid-preferring gate
is the CGP expert, seeds 42/43/44). Run from the `ICLR_review/` directory
(kernel: the `deepfusion` conda environment).

In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
import torch
import anndata

ROOT = Path("..")
IFNB = ROOT / "ifnb"
EXP02 = Path("exp02_expert_masking")
EXP05 = Path("exp05_random_grouping")
OUT = Path("exp08_e7cp_controls")
OUT.mkdir(exist_ok=True)

MYELOID = ["DC", "CD14 Mono", "CD16 Mono"]
LYMPHOID = ["CD4 Naive T", "CD4 Memory T", "CD8 T", "T activated",
            "B", "B Activated", "NK"]

In [2]:
mm = torch.load(EXP02 / "module_matrix_4440_rebuilt.pt", map_location="cpu").numpy()
vm = pickle.load(open(IFNB / "enhanced_results/DeepFusion/cache/valid_modules_cache_dd0.5.pkl", "rb"))
module_names = list(vm.keys())

ad = anndata.read_h5ad(IFNB / "enhanced_results/results/step5_add_DeepFusion.h5ad")
labels = ad.obs["seurat_annotations"].astype(str).values

id2name = pd.read_csv(IFNB / "SC_data/c2.all.v2026.1.Hs.symbols_ID_to_category_name_filtered_parent.csv")
id2name = dict(zip(id2name["labels"].astype(str), id2name["name"]))
gmap = json.load(open(IFNB / "SC_data/module_groups_v2026.json"))
groups = np.array([gmap[k] for k in module_names])

def pretty(mname):
    return id2name.get(mname.replace("Module_", ""), mname)

myel = np.isin(labels, MYELOID).astype(float)
m_myel, m_lymph = np.isin(labels, MYELOID), np.isin(labels, LYMPHOID)

def corr(a, b):
    if a.std() < 1e-12 or b.std() < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def partial_corr(x, y, c):
    C = np.column_stack([np.ones_like(c), c])
    rx = x - C @ np.linalg.lstsq(C, x, rcond=None)[0]
    ry = y - C @ np.linalg.lstsq(C, y, rcond=None)[0]
    return corr(rx, ry)

def analyze_gate(w_gate, tag):
    """Correlate one gate's per-cell weight with all 4,440 module activities."""
    rs = np.array([corr(mm[:, i], w_gate) for i in range(mm.shape[1])])
    rp = np.array([partial_corr(mm[:, i], w_gate, myel) for i in range(mm.shape[1])])
    res = pd.DataFrame({"module": module_names,
                        "name": [pretty(m) for m in module_names],
                        "group": groups, "r": rs, "r_partial_label": rp,
                        "run": tag})
    return res

## Real model: the CGP gate (three seeds)

In [3]:
real = []
for s, k in [("seed42", "X_expert_weights_40_std"),
             ("seed43", "X_expert_weights_40_std_s43"),
             ("seed44", "X_expert_weights_40_std_s44")]:
    w_cgp = np.asarray(ad.obsm[k])[:, 0]
    res = analyze_gate(w_cgp, f"real_CGP_{s}")
    real.append(res)
    top = res.nlargest(5, "r")
    print(f"real {s}: max r = {res.r.max():.3f}, top-5: "
          + "; ".join(f"{n} ({r:.3f})" for n, r in zip(top.name, top.r)))
real = pd.concat(real)

real seed42: max r = 0.873, top-5: FULCHER_INFLAMMATORY_RESPONSE_LECTIN_VS_LPS_DN (0.873); VERHAAK_AML_WITH_NPM1_MUTATED_UP (0.866); GAL_LEUKEMIC_STEM_CELL_DN (0.859); RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP (0.845); RUTELLA_RESPONSE_TO_HGF_UP (0.835)


real seed43: max r = 0.882, top-5: VERHAAK_AML_WITH_NPM1_MUTATED_UP (0.882); GAL_LEUKEMIC_STEM_CELL_DN (0.868); RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP (0.862); DELYS_THYROID_CANCER_UP (0.853); LENAOUR_DENDRITIC_CELL_MATURATION_DN (0.848)


real seed44: max r = 0.895, top-5: VERHAAK_AML_WITH_NPM1_MUTATED_UP (0.895); GAL_LEUKEMIC_STEM_CELL_DN (0.884); RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP (0.870); FULCHER_INFLAMMATORY_RESPONSE_LECTIN_VS_LPS_DN (0.870); DELYS_THYROID_CANCER_UP (0.860)


## Random-grouping models: the myeloid-preferring gate of each run

In [4]:
rand = []
gate_info = []
for rep in (1, 2, 3):
    for seed in (42, 43):
        run = f"rep{rep}_seed{seed}"
        z = np.load(EXP05 / "runs" / run / "embeddings.npz", allow_pickle=True)
        w = z["X_expert_weights"]              # [N, 3] = [RAND_A, RAND_B, gene]
        assert (z["label"].astype(str) == labels).all(), "cell order mismatch"
        dA = w[m_myel, 0].mean() - w[m_lymph, 0].mean()
        dB = w[m_myel, 1].mean() - w[m_lymph, 1].mean()
        gate = 0 if dA >= dB else 1
        gate_info.append({"run": run, "chosen_gate": "RAND_A" if gate == 0 else "RAND_B",
                          "delta_myel_minus_lymph": max(dA, dB)})
        res = analyze_gate(w[:, gate], run)
        rand.append(res)
        top = res.nlargest(5, "r")
        print(f"{run}: gate={'A' if gate==0 else 'B'} (delta={max(dA,dB):.3f}), "
              f"max r = {res.r.max():.3f}, top-5: "
              + "; ".join(f"{n} ({r:.3f})" for n, r in zip(top.name, top.r)))
rand = pd.concat(rand)
pd.DataFrame(gate_info).to_csv(OUT / "control2_random_gate_choice.csv", index=False)
real.to_csv(OUT / "control2_real_CGP_correlations.csv", index=False)
rand.to_csv(OUT / "control2_random_gate_correlations.csv", index=False)

rep1_seed42: gate=A (delta=0.709), max r = 0.892, top-5: VERHAAK_AML_WITH_NPM1_MUTATED_UP (0.892); GAL_LEUKEMIC_STEM_CELL_DN (0.887); RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP (0.866); DELYS_THYROID_CANCER_UP (0.858); LENAOUR_DENDRITIC_CELL_MATURATION_DN (0.855)


rep1_seed43: gate=B (delta=0.478), max r = 0.867, top-5: VERHAAK_AML_WITH_NPM1_MUTATED_UP (0.867); GAL_LEUKEMIC_STEM_CELL_DN (0.846); RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP (0.831); LENAOUR_DENDRITIC_CELL_MATURATION_DN (0.823); DELYS_THYROID_CANCER_UP (0.815)


rep2_seed42: gate=B (delta=0.496), max r = 0.822, top-5: VERHAAK_AML_WITH_NPM1_MUTATED_UP (0.822); GAL_LEUKEMIC_STEM_CELL_DN (0.808); RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP (0.792); LENAOUR_DENDRITIC_CELL_MATURATION_DN (0.789); DELYS_THYROID_CANCER_UP (0.776)


rep2_seed43: gate=B (delta=0.505), max r = 0.814, top-5: VERHAAK_AML_WITH_NPM1_MUTATED_UP (0.814); LENAOUR_DENDRITIC_CELL_MATURATION_DN (0.801); GAL_LEUKEMIC_STEM_CELL_DN (0.794); RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP (0.788); DELYS_THYROID_CANCER_UP (0.778)


rep3_seed42: gate=B (delta=0.416), max r = 0.704, top-5: VERHAAK_AML_WITH_NPM1_MUTATED_UP (0.704); GAL_LEUKEMIC_STEM_CELL_DN (0.679); LENAOUR_DENDRITIC_CELL_MATURATION_DN (0.662); DELYS_THYROID_CANCER_UP (0.651); RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP (0.650)


rep3_seed43: gate=B (delta=0.642), max r = 0.825, top-5: LENAOUR_DENDRITIC_CELL_MATURATION_DN (0.825); VERHAAK_AML_WITH_NPM1_MUTATED_UP (0.823); DELYS_THYROID_CANCER_UP (0.809); GAL_LEUKEMIC_STEM_CELL_DN (0.804); RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP (0.796)


## Summary — real CGP gate vs six random myeloid-preferring gates

In [5]:
def summarize(df_all):
    out = []
    for run, g in df_all.groupby("run"):
        top10 = g.nlargest(10, "r")
        out.append({
            "run": run,
            "max r": g.r.max(),
            "mean top-10 r": top10.r.mean(),
            "max partial r | label": g.r_partial_label.max(),
            "mean top-10 partial r": g.nlargest(10, "r_partial_label").r_partial_label.mean(),
            "top-15 that are CGP modules (%)": 100 * (g.nlargest(15, "r").group == "CGP").mean(),
        })
    return pd.DataFrame(out)

summ = pd.concat([summarize(real), summarize(rand)]).reset_index(drop=True)
print(summ.round(3).to_string(index=False))
summ.to_csv(OUT / "control2_summary.csv", index=False)

            run  max r  mean top-10 r  max partial r | label  mean top-10 partial r  top-15 that are CGP modules (%)
real_CGP_seed42  0.873          0.842                  0.652                  0.607                            100.0
real_CGP_seed43  0.882          0.852                  0.595                  0.580                            100.0
real_CGP_seed44  0.895          0.863                  0.657                  0.629                            100.0
    rep1_seed42  0.892          0.856                  0.574                  0.557                            100.0
    rep1_seed43  0.867          0.821                  0.483                  0.465                            100.0
    rep2_seed42  0.822          0.777                  0.378                  0.354                            100.0
    rep2_seed43  0.814          0.775                  0.427                  0.404                            100.0
    rep3_seed42  0.704          0.642                  0.358    

## Top-signature overlap

Do the random gates point at the same signatures as the CGP gate?
Jaccard overlap of each run's top-15 with the real seed-42 top-15.

In [6]:
real_top15 = set(real[real.run == "real_CGP_seed42"].nlargest(15, "r").name)
for run, g in rand.groupby("run"):
    t = set(g.nlargest(15, "r").name)
    inter = len(real_top15 & t)
    print(f"{run}: overlap with real CGP top-15 = {inter}/15")
print("\nReal CGP top-15 (seed 42):")
print("\n".join(sorted(real_top15)))

rep1_seed42: overlap with real CGP top-15 = 11/15
rep1_seed43: overlap with real CGP top-15 = 12/15
rep2_seed42: overlap with real CGP top-15 = 11/15
rep2_seed43: overlap with real CGP top-15 = 11/15
rep3_seed42: overlap with real CGP top-15 = 9/15
rep3_seed43: overlap with real CGP top-15 = 12/15

Real CGP top-15 (seed 42):
BLANCO_MELO_BRONCHIAL_EPITHELIAL_CELLS_INFLUENZA_A_DEL_NS1_INFECTION_UP
DELYS_THYROID_CANCER_UP
DIAZ_CHRONIC_MYELOGENOUS_LEUKEMIA_DN
FULCHER_INFLAMMATORY_RESPONSE_LECTIN_VS_LPS_DN
GAL_LEUKEMIC_STEM_CELL_DN
JISON_SICKLE_CELL_DISEASE_UP
LENAOUR_DENDRITIC_CELL_MATURATION_DN
MARKEY_RB1_ACUTE_LOF_DN
MARKEY_RB1_CHRONIC_LOF_DN
MCLACHLAN_DENTAL_CARIES_UP
MULLIGHAN_MLL_SIGNATURE_2_UP
RUTELLA_RESPONSE_TO_CSF2RB_AND_IL4_DN
RUTELLA_RESPONSE_TO_HGF_UP
RUTELLA_RESPONSE_TO_HGF_VS_CSF2RB_AND_IL4_UP
VERHAAK_AML_WITH_NPM1_MUTATED_UP
